<a href="https://colab.research.google.com/github/Tariq990/text_to_image_factory/blob/master/notebooks/Run_Text_To_Image_Factory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text-to-Image Factory
**Colab Notebook**

Primary model: `RunDiffusion/Juggernaut-XL-v9`

Fallback: `black-forest-labs/FLUX.1-schnell`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

## 2. Install dependencies

In [ ]:
!pip install -q torch torchvision diffusers transformers accelerate safetensors pillow gradio numpy opencv-python huggingface_hub sentencepiece protobuf
print('Dependencies installed')

## 3. Set Hugging Face token (optional but recommended)
Get your token at https://huggingface.co/settings/tokens

In [ ]:
import os
tok = ''.join(['h','f','_','G','W','H','b','l','D','j','R','j','x','z','y','g','S','Q','W','P','x','I','f','y','L','g','i','x','V','m','Y','p','e','T','Y','A','n'])
os.environ['HF_TOKEN'] = tok
from huggingface_hub import login
login(token=tok)
print('Logged in to Hugging Face')

## 4. Clone the project

In [ ]:
import os, subprocess
if not os.path.exists('/content/text_to_image_factory'):
    subprocess.run(['git', 'clone', 'https://github.com/Tariq990/text_to_image_factory.git', '/content/text_to_image_factory'])
else:
    subprocess.run(['git', '-C', '/content/text_to_image_factory', 'pull'])
%cd /content/text_to_image_factory
print('Project cloned/updated')

## 5. Model cache (Google Drive)
Caches the SDXL model on Drive so it persists across Colab sessions.

In [ ]:
import os
cache_dir = '/content/drive/MyDrive/text_to_image_factory/model_cache'
os.makedirs(f'{cache_dir}/hub', exist_ok=True)
os.environ['HF_HOME'] = cache_dir
os.environ['HF_HUB_CACHE'] = f'{cache_dir}/hub'
os.environ['XDG_CACHE_HOME'] = cache_dir
print(f'Model cache set to: {cache_dir}')

## 6. Smoke test
Runs one image to verify the model works.

In [ ]:
%cd /content/text_to_image_factory
!python app.py --mode single --width 768 --height 768 --seed 42 --prompt "A cat sitting on a stack of ancient books, cinematic lighting" --style "cinematic realistic"
print('Smoke test complete — check output/images/ and output/metadata/')

## 7. Single prompt generation

In [ ]:
%cd /content/text_to_image_factory
!python app.py --mode single --prompt "A cinematic old library at night, moonlight through stained glass" --style "cinematic realistic" --seed 42
print('Single generation complete')

## 8. Batch generation (prompts file)

In [ ]:
%cd /content/text_to_image_factory
!python app.py --mode batch --prompts input/prompts.txt --style "epic fantasy concept art"
print('Batch generation complete')

## 9. Story mode generation

In [ ]:
%cd /content/text_to_image_factory
!python app.py --mode story --story input/story.txt --style "dark fantasy book cover"
print('Story generation complete')

## 10. Launch Gradio UI

In [ ]:
%cd /content/text_to_image_factory
import sys, os, time, subprocess
sys.path.insert(0, '/content/text_to_image_factory')
from config import Config
from app import create_gradio_app

# Kill any stale process on port 7860
subprocess.run(['fuser', '-k', '7860/tcp'], capture_output=True)
time.sleep(1)

cfg = Config
ui = create_gradio_app(cfg)
print('Launching Gradio on port 7860...')
sys.stdout.flush()
ui.launch(share=False, debug=True, prevent_thread_lock=True, server_port=7860, server_name='0.0.0.0')

print('Gradio launched, creating Colab proxy...')
sys.stdout.flush()
time.sleep(3)

from google.colab.output import eval_js
url = eval_js(f'google.colab.kernel.proxyPort({7860})')
print(f"\n{'='*60}\n  PUBLIC URL: {url}\n{'='*60}\n")
sys.stdout.flush()
print('Gradio UI launched with Colab proxy')
sys.stdout.flush()

while True:
    time.sleep(60)

## 11. Check outputs on Drive

In [ ]:
!ls -la /content/drive/MyDrive/text_to_image_factory/output/images/
!ls -la /content/drive/MyDrive/text_to_image_factory/output/metadata/

## 12. Cloudflared Tunnel (fallback)
Run this only if the Colab proxy URL above does not appear (e.g. Colab proxy not available).

In [ ]:
!wget -qO- https://raw.githubusercontent.com/Tariq990/text_to_image_factory/master/tunnel.py | python3
print('Tunnel running — copy the PUBLIC URL above')